# 15 — Final calibrated ensemble: saved test results

The selected notebook-13 model is a fixed 50/50 average of sigmoid-calibrated MLP and Extra Trees scores. Final fitting, export and one fixed test evaluation run on the cluster using `cluster/final_ensemble_test.slurm`. See `cluster/FINAL_ENSEMBLE_README.md`.

This notebook **never trains, downloads or predicts**. Download the saved `test_metrics.json` and model ZIP after the cluster job completes. The model package can be attached to a GitHub Release; teammates can load it without retraining.

Current train/test ID and group separation passed audit. Historical project-wide test use remains unverified, so these results must be described with that limitation. No test result has been supplied or invented in this notebook.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/d1_final_ensemble.py').is_file())
RESULTS = ROOT / 'data/processed/final_ensemble_test/v1'
path = RESULTS / 'test_metrics.json'
result = json.loads(path.read_text()) if path.exists() else None
if result is None:
    print('No final results yet. Submit the cluster job and download its saved metrics.')
else:
    print(result['model'])
    print('Test rows:', result['rows'], '| frauds:', result['fraudulent'], '| groups:', result['groups'])
    display(pd.DataFrame([result['metrics']]).round(4))
    print('Threshold was selected on development inner OOF predictions, not test.')


In [ ]:
if result is not None:
    intervals = result['bootstrap']['intervals']
    display(pd.DataFrame([{'metric': name, 'test_estimate': result['metrics'][name],
                           'group_bootstrap_low': bounds['low'],
                           'group_bootstrap_high': bounds['high']}
                          for name, bounds in intervals.items()]).round(4))
    print('Bootstrap valid/requested:', result['bootstrap']['valid'], '/', result['bootstrap']['requested'])
    print(result['history_note'])
    print('Model archive:', result.get('artifact_archive'))
    print('ZIP SHA256:', result.get('artifact_archive_sha256'))


## Interpretation

Report these as the fixed selected pipeline's test results, not a new model-selection round. Group bootstrap intervals describe uncertainty of this model's metrics; they are not evidence of superiority over an unevaluated comparator. Do not adjust the model or cutoff after reading test scores. Agent integration/evaluation is a separate step.
